# 🧐 The Reflection Architecture: Generate → Critique → Refine

## Learning Objectives
In this notebook, you will learn:
1. **The architecture** - Three roles, three structured outputs: a generator, a critic and a refiner
2. **A structured critique** - `has_errors`, `is_efficient` and concrete suggestions instead of a free-text review
3. **The workflow as a graph** - A linear LangGraph pipeline whose state holds the Pydantic objects themselves
4. **Measuring the effect** - Score the draft and the refined code with hidden tests, in code, over several runs
5. **Self-bias** - Compare what the critic *believed* about the draft with what the tests found

## Where this fits
`01_reflection_agents.ipynb` looped a free-text critique; `02_reflexion_agents.ipynb` grounded it in web search. This notebook is the smallest structured version - one pass, typed outputs - applied to code, where the result can be **tested** instead of judged. Module 09's evaluator-optimizer (`modules/09_evaluator_optimizer/notebooks/01_evaluator_optimizer_langgraph.ipynb`) is the looping cousin; `07_reflective_code_generation_agent.ipynb` closes the gap this notebook exposes by running the code during the loop.

## Prerequisites
- `with_structured_output` - `modules/04_langchain_agents_and_middleware/notebooks/03_structured_output.ipynb`
- Dependencies installed with `uv sync` at the repository root
- `OPENAI_API_KEY` in the project-root `.env`

> **Note**: Part 5 runs model-written code with `exec` on your machine. That is fine for this small, pure-Python task; for untrusted code use a sandbox such as the hosted interpreter in `modules/02_tools_and_function_calling/notebooks/06_hosted_code_execution.ipynb`.

---
## 📖 Part 1: The Reflection Architecture

This pattern elevates a Large Language Model (LLM) from a simple, single-pass generator into a more deliberate and robust reasoner. Instead of just providing the first answer it comes up with, a reflective agent takes a step back to critique, analyze, and refine its own work. This iterative process of self-improvement is a cornerstone of building more reliable and higher-quality AI systems.

| Property | Value |
|---|---|
| Origin | Madaan et al., *Self-Refine: Iterative Refinement with Self-Feedback*, NeurIPS 2023 ([arXiv:2303.17651](https://arxiv.org/abs/2303.17651)) |

### Definition
The **Reflection** architecture involves an agent critiquing and revising its own output before returning a final answer. Instead of a single-pass generation, it engages in a multi-step internal monologue: produce, evaluate, and improve. This mimics the human process of drafting, reviewing, and editing to catch errors and enhance quality.

### High-level Workflow

1.  **Generate:** The agent produces an initial draft or solution based on the user's prompt.
2.  **Critique:** The agent then switches roles to become a critic. It asks itself questions like: *"What could be wrong with this answer?"*, *"What is missing?"*, *"Is this solution optimal?"*, or *"Are there any logical flaws or bugs?"*.
3.  **Refine:** Using the insights from its self-critique, the agent generates a final, improved version of the output.

### When to Use / Applications
*   **Code Generation:** The initial code might have bugs, be inefficient, or lack comments. Reflection allows the agent to act as its own code reviewer, catching errors and improving style before presenting the final script.
*   **Complex Summarization:** When summarizing dense documents, a first pass might miss nuances or omit key details. A reflection step helps ensure the summary is comprehensive and accurate.
*   **Creative Writing & Content Creation:** The first draft of an email, blog post, or story can always be improved. Reflection allows the agent to refine its tone, clarity, and impact.

### Strengths & Weaknesses
*   **Strengths:**
    *   **Improved Quality:** Directly addresses and corrects errors, leading to more accurate, robust, and well-reasoned outputs.
    *   **Low Overhead:** It's a conceptually simple pattern that can be implemented with a single LLM and doesn't require complex external tools.
*   **Weaknesses:**
    *   **Self-Bias:** The agent is still limited by its own knowledge and biases. If it doesn't know a better way to solve a problem, it can't critique its way to a better solution. It can fix flaws it recognizes but can't invent knowledge it lacks.
    *   **Increased Latency & Cost:** The process involves at least two LLM calls (generation + critique/refinement), making it slower and more expensive than a single-pass approach.

---
## 📦 Step 1: Environment Setup

Load the API key and create the models: a generator with some randomness (so repeated runs produce different drafts to measure), and a deterministic critic and refiner.

In [ ]:
# ============================================================================
# ENVIRONMENT SETUP: Load API keys and suppress warnings
# ============================================================================
import warnings

from dotenv import find_dotenv, load_dotenv

warnings.filterwarnings("ignore")

# Reads OPENAI_API_KEY from the project-root .env
load_dotenv(find_dotenv(usecwd=True))

print("✅ Environment ready")

In [ ]:
# ============================================================================
# LLM INITIALIZATION: Generator (varied drafts) + critic/refiner (deterministic)
# ============================================================================
from aied_helpers import get_openai_llm

generator_llm = get_openai_llm(model="gpt-4o-mini", temperature=0.7)
llm = get_openai_llm(model="gpt-4o-mini")

print(f"✅ Generator: {generator_llm.model_name} (temperature={generator_llm.temperature})")
print(f"✅ Critic/refiner: {llm.model_name} (temperature={llm.temperature})")

---
## 🧱 Part 2: Three Structured Outputs

Pydantic models act as a contract for each role. The `Critique` model is the important one: asking for specific fields like `has_errors` and `is_efficient` guides the LLM to a more structured evaluation than just "review the code" - and gives us a claim we can check later.

In [ ]:
# ============================================================================
# SCHEMAS: DraftCode -> Critique -> RefinedCode
# ============================================================================
from typing import List, Optional

from pydantic import BaseModel, Field

class DraftCode(BaseModel):
    """Schema for the initial code draft generated by the agent."""
    code: str = Field(description="The Python code generated to solve the user's request.")
    explanation: str = Field(description="A brief explanation of how the code works.")

class Critique(BaseModel):
    """Schema for the self-critique of the generated code."""
    has_errors: bool = Field(description="Does the code have any potential bugs or logical errors?")
    is_efficient: bool = Field(description="Is the code written in an efficient and optimal way?")
    suggested_improvements: List[str] = Field(description="Specific, actionable suggestions for improving the code.")
    critique_summary: str = Field(description="A summary of the critique.")

class RefinedCode(BaseModel):
    """Schema for the final, refined code after incorporating the critique."""
    refined_code: str = Field(description="The final, improved Python code.")
    refinement_summary: str = Field(description="A summary of the changes made based on the critique.")


print(f"✅ Critique fields: {list(Critique.model_fields)}")

---
## ⚙️ Part 3: Generator, Critic and Refiner Nodes

Each node reads from the state and writes one structured object back. The state holds the **Pydantic objects themselves** (`draft.code`, `critique.suggested_improvements`) rather than dicts - an earlier variant of this notebook converted them with `.model_dump()` at every step, which only adds string keys to get wrong.

The task is deliberately under-specified: a duration parser whose request says nothing about malformed input, spaces, or repeated units. A careful reviewer should think of those; that is what the critic is for.

In [ ]:
# ============================================================================
# STATE + NODES: generator -> critic -> refiner
# ============================================================================
from typing_extensions import TypedDict


class ReflectionState(TypedDict):
    """Represents the state of our reflection graph."""
    user_request: str
    draft: Optional[DraftCode]
    critique: Optional[Critique]
    refined_code: Optional[RefinedCode]


def generator_node(state: ReflectionState):
    """Generates the initial draft of the code."""
    prompt = f"""You are an expert Python programmer. Write a Python function to solve the following request.
Provide a simple, clear implementation and an explanation.

Request: {state['user_request']}
"""
    return {"draft": generator_llm.with_structured_output(DraftCode).invoke(prompt)}


def critic_node(state: ReflectionState):
    """Critiques the generated code for errors and inefficiencies."""
    prompt = f"""You are an expert code reviewer and senior Python developer. Your task is to perform a thorough critique of the following code.

Analyze the code for:
1.  **Bugs and Errors:** Are there any potential runtime errors, logical flaws, or edge cases that are not handled?
2.  **Efficiency and Best Practices:** Is this the most efficient way to solve the problem? Does it follow standard Python conventions (PEP 8)?

Provide a structured critique with specific, actionable suggestions.

Request the code was written for: {state['user_request']}

Code to Review:
```python
{state['draft'].code}
```
"""
    return {"critique": llm.with_structured_output(Critique).invoke(prompt)}


def refiner_node(state: ReflectionState):
    """Refines the code based on the critique."""
    prompt = f"""You are an expert Python programmer tasked with refining a piece of code based on a critique.

Your goal is to rewrite the original code, implementing all the suggested improvements from the critique.

**Original Code:**
```python
{state['draft'].code}
```

**Critique and Suggestions:**
{state['critique'].model_dump_json(indent=2)}

Please provide the final, refined code and a summary of the changes you made.
"""
    return {"refined_code": llm.with_structured_output(RefinedCode).invoke(prompt)}


print("✅ Nodes defined")

---
## 🏗️ Part 4: The Workflow as a Graph

The workflow is a simple linear sequence: **Generate → Critique → Refine**. There is no loop - one reflection pass.

In [ ]:
# ============================================================================
# GRAPH: generator -> critic -> refiner -> END
# ============================================================================
from langgraph.graph import END, START, StateGraph

graph_builder = StateGraph(ReflectionState)
graph_builder.add_node("generator", generator_node)
graph_builder.add_node("critic", critic_node)
graph_builder.add_node("refiner", refiner_node)
graph_builder.add_edge(START, "generator")
graph_builder.add_edge("generator", "critic")
graph_builder.add_edge("critic", "refiner")
graph_builder.add_edge("refiner", END)
reflection_app = graph_builder.compile()

print("🧩 Nodes:", [n for n in reflection_app.get_graph().nodes if not n.startswith("__")])

### 4.1 🚀 Run It and Read the Before and After

In [ ]:
# ============================================================================
# RUN: One pass, then print draft, critique and refined code
# ============================================================================
user_request = "Write a Python function `parse_duration(text)` that converts a duration string such as '1h30m', '45s' or '2h5m10s' into a total number of seconds."
final_state = reflection_app.invoke({"user_request": user_request})

print("\n=== INITIAL DRAFT ===")
print(final_state["draft"].code)
print("\n=== CRITIQUE ===")
print(f"has_errors={final_state['critique'].has_errors}, is_efficient={final_state['critique'].is_efficient}")
print(final_state["critique"].critique_summary)
for improvement in final_state["critique"].suggested_improvements:
    print(f"  - {improvement}")
print("\n=== REFINED CODE ===")
print(final_state["refined_code"].refinement_summary)
print(final_state["refined_code"].refined_code)

---
## 📏 Part 5: Measuring the Effect

The original version of this notebook asked another LLM to score both versions from 1 to 10. For code we can do better: **run it**. The hidden tests below encode what a reasonable user expects of a duration parser - valid inputs, surrounding spaces, and a `ValueError` for malformed input. Neither the generator nor the critic ever sees them.

In [ ]:
# ============================================================================
# HIDDEN TESTS: What "correct" means, checked by running the code
# ============================================================================
HIDDEN_TESTS = [
    ("1h30m", 5400), ("45s", 45), ("2h5m10s", 7510), ("90m", 5400), (" 2h ", 7200),
    ("abc", ValueError), ("", ValueError), ("5x", ValueError), ("1h1h", ValueError), ("h", ValueError),
]


def run_hidden_tests(source: str) -> tuple[int, list[str]]:
    """Exec the code in a fresh namespace and return (passed, failing inputs)."""
    namespace = {}
    try:
        exec(source, namespace)
        parse_duration = namespace["parse_duration"]
    except Exception as exc:
        return 0, [f"could not load: {type(exc).__name__}"]
    passed, failed = 0, []
    for text, expected in HIDDEN_TESTS:
        try:
            ok = parse_duration(text) == expected
        except Exception as exc:
            ok = expected is ValueError and isinstance(exc, ValueError)
        passed += ok
        if not ok:
            failed.append(repr(text))
    return passed, failed


for label, source in [("draft", final_state["draft"].code), ("refined", final_state["refined_code"].refined_code)]:
    passed, failed = run_hidden_tests(source)
    print(f"{label:>8}: {passed}/{len(HIDDEN_TESTS)} passed, failing: {failed}")

One run is an anecdote. Run the whole workflow three more times and compare the critic's **belief** (`has_errors`) with the draft's **measured** result:

In [ ]:
# ============================================================================
# REPEATED RUNS: Draft vs. refined pass counts, and what the critic believed
# ============================================================================
print(f"{'run':>4} {'draft':>7} {'refined':>8}  critic said has_errors")
for run in range(1, 4):
    state = reflection_app.invoke({"user_request": user_request})
    draft_passed, _ = run_hidden_tests(state["draft"].code)
    refined_passed, _ = run_hidden_tests(state["refined_code"].refined_code)
    print(f"{run:>4} {draft_passed:>5}/{len(HIDDEN_TESTS)} {refined_passed:>6}/{len(HIDDEN_TESTS)}  {state['critique'].has_errors}")

Read the table. Refinement helps when the critique happens to name what the tests check ("validate the input", "raise a `ValueError` for unknown units"). It is not reliable, though: across runs the refined code can pass the same number of tests as the draft, or even fewer - a refiner that "simplifies" the parser with a regular expression may quietly drop validation. And `has_errors` is an opinion formed without running anything: in our test runs the critic sometimes answered `False` for drafts that failed half the tests.

That is the **self-bias** weakness from Part 1 made measurable: the critic can only flag problems it can imagine, it cannot run the code, and nothing checks that the refinement is actually better. Two remedies follow in this module - independent verification questions (`04_chain_of_verification.ipynb`) and critique grounded in real execution (`07_reflective_code_generation_agent.ipynb`).

---
## 📝 Summary

In this notebook, we learned:

### 1. The architecture
- Generator, critic and refiner are three prompts with three structured outputs; the whole workflow is a linear graph.

### 2. A structured critique
- Typed fields (`has_errors`, `is_efficient`, `suggested_improvements`) make the critique inspectable and easy to pass to the refiner.

### 3. The workflow as a graph
- Keep Pydantic objects in state and read them with attributes - no dict round-trips.

### 4. Measuring the effect
- For code, hidden tests run in code beat an LLM's 1-10 score. Measure over several runs: one run is an anecdote.

### 5. Self-bias
- The critic's self-assessment is not evidence. Reflection improves what the critic can imagine; ground it in execution or independent checks for more.

### Next Steps
- **`04_chain_of_verification.ipynb`**: verify factual claims with independent questions that never see the original answer